# Representation metrics: similarity × communication (first slice)

This notebook computes PCA-based **state dimensionality** metrics and transfer metrics for a small set of two-module RNN runs, focusing on:

- **Similarity**: same / near / far geometry participants (`geom_sub_*`).
- **Communication**: low vs high sparsity runs with identical architecture and routing.

It uses helpers from `a1b2.analysis.transfer_interference` to summarize:

- variance explained by the first 2 PCs (`var_topk`)
- number of PCs needed to reach 90% and 99% variance (`n_pcs_90`, `n_pcs_99`)
- (optionally) hidden-state drift between phases.

This notebook is intended as the main analysis entry point for the "first slice" experiments before adding init_scale and further factors.


In [1]:
# 1. Setup and paths

import sys
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path(os.getcwd()).resolve()
while not (project_root / "a1b2").exists():
    if project_root == project_root.parent:
        raise RuntimeError("Project root (containing a1b2) not found.")
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from a1b2.utils.run_config import build_run_id
from a1b2.analysis import transfer_interference as ann

data_folder = project_root / "data"
sim_folder = data_folder / "simulations"
config_path = project_root / "a1b2" / "models" / "experiments.json"
with open(config_path, "r") as f:
    settings = json.load(f)

print("Project root:", project_root)
print("Simulations folder:", sim_folder)


Project root: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular
Simulations folder: /home/kat/workspace/Structure-Function-Analysis-of-Network-Topologies/a1b2_modular/data/simulations


In [2]:
# 2. Select runs for first slice (similarity × communication)

# Example: two-module RNN, dim_hidden=25, nb_steps=2, shared routing, common_readout=True
TARGET_ARCH = "two_module_rnn"
TARGET_DIM_HIDDEN = 25
TARGET_NB_STEPS = 2
TARGET_ROUTING = "shared"  # keep routing fixed for the first slice
TARGET_COMMON_READOUT = True
SPARSITY_LEVELS = [0.3, 1.0]  # low vs high communication


def want_condition_first_slice(c):
    if c.get("arch") != TARGET_ARCH:
        return False
    if c.get("dim_hidden") != TARGET_DIM_HIDDEN:
        return False
    if c.get("nb_steps", 1) != TARGET_NB_STEPS:
        return False
    if c.get("common_readout", True) is not TARGET_COMMON_READOUT:
        return False
    if c.get("input_routing", "shared") != TARGET_ROUTING:
        return False
    if c.get("sparsity") not in SPARSITY_LEVELS:
        return False
    return True


selected_conditions = [c for c in settings["conditions"] if want_condition_first_slice(c)]
run_ids_with_paths = []
for c in selected_conditions:
    run_id = build_run_id(c)
    path = sim_folder / run_id
    run_ids_with_paths.append((run_id, path if path.exists() else None))

print("Selected runs (first slice):")
for run_id, path in run_ids_with_paths:
    status = "OK" if path is not None else "missing"
    print(f"  {run_id}: {status}")


Selected runs (first slice):
  two_module_rnn_25_nb2_nb2_shared_sp1.0_sep_cr_RNN: OK
  two_module_rnn_25_low_sparse_nb2_nb2_shared_sp0.3_sep_cr_RNN: OK
  two_module_rnn_25_nb2_init0.1_nb2_shared_sp1.0_sep_cr_RNN_init0.1: missing
  two_module_rnn_25_nb2_init0.01_nb2_shared_sp1.0_sep_cr_RNN_init0.01: missing


In [ ]:
# 3. Load ANN/RNN data for selected runs

all_ann_data = {}
for run_id, path in run_ids_with_paths:
    if path is None:
        continue
    ann_data = ann.load_ann_data(str(path), load_rnn_extra=True)
    if len(ann_data["same"]) > 0 and len(ann_data["near"]) > 0 and len(ann_data["far"]) > 0:
        all_ann_data[run_id] = ann_data

run_labels = list(all_ann_data.keys())
print("Loaded", len(run_labels), "runs with all three schedules.")
for rid in run_labels:
    d = all_ann_data[rid]
    print(f"  {rid}: same={len(d['same'])} near={len(d['near'])} far={len(d['far'])}")


In [ ]:
# 4. Compute representation metrics (state dimensionality)

rows = []
for run_id in run_labels:
    data = all_ann_data[run_id]
    # Last-step state dimensionality; includes combined/core/comms when present
    df_rep = ann.compute_pca_representation_metrics(
        data,
        variance_thresholds=(0.9, 0.99),
        top_k=2,
        include_paths=("combined", "core", "comms"),
    )
    df_rep["run_id"] = run_id
    rows.append(df_rep)

if rows:
    rep_metrics = pd.concat(rows, ignore_index=True)
else:
    rep_metrics = pd.DataFrame()

rep_metrics.head()


In [ ]:
# 5. Compute transfer and (optional) drift metrics

transfer_rows = []
drift_rows = []

for run_id in run_labels:
    data = all_ann_data[run_id]

    # Transfer metrics per schedule
    df_transfer = ann.compute_transfer_anns(data)
    df_transfer["run_id"] = run_id
    transfer_rows.append(df_transfer)

    # Hidden-state drift metrics (pre→A1 and A1→B) for geometry participants
    for sched in ["same", "near", "far"]:
        for entry in data[sched]:
            participant_id = str(entry["participant"])
            pre = entry.get("hiddens_pre_training")
            post_A = entry.get("hiddens_post_phase_0")
            post_B = entry.get("hiddens_post_phase_1")
            if pre is None or post_A is None or post_B is None:
                continue
            drift_preA = ann.compute_hidden_drift(pre, post_A)
            drift_AB = ann.compute_hidden_drift(post_A, post_B)
            drift_rows.append({
                "run_id": run_id,
                "participant": participant_id,
                "condition": sched,
                "drift_preA": drift_preA,
                "drift_AB": drift_AB,
            })

transfer_metrics = pd.concat(transfer_rows, ignore_index=True) if transfer_rows else pd.DataFrame()
drift_metrics = pd.concat(drift_rows, ignore_index=True) if drift_rows else pd.DataFrame()

transfer_metrics.head(), drift_metrics.head()


In [ ]:
# 6. Aggregate and preview summary tables

if not rep_metrics.empty:
    # Example aggregation: mean over participants per run/schedule/phase/pathway
    rep_summary = (
        rep_metrics
        .groupby(["run_id", "condition", "phase", "pathway"], as_index=False)
        .agg({"var_topk": "mean", "n_pcs_90": "mean", "n_pcs_99": "mean"})
    )
    display(rep_summary.head())
else:
    print("No representation metrics computed.")

if not transfer_metrics.empty:
    transfer_summary = (
        transfer_metrics
        .groupby(["run_id", "condition"], as_index=False)
        .agg({"error_diff": "mean"})
    )
    display(transfer_summary.head())
else:
    print("No transfer metrics computed.")

if not drift_metrics.empty:
    drift_summary = (
        drift_metrics
        .groupby(["run_id", "condition"], as_index=False)
        .agg({"drift_preA": "mean", "drift_AB": "mean"})
    )
    display(drift_summary.head())
else:
    print("No drift metrics computed.")


## 7. Next steps

- Use these summary tables to generate plots comparing **similarity** (same/near/far) and **communication** (low vs high sparsity) for each pathway (combined/core/comms).
- Once init_scale runs are available (e.g. run_ids including `init0.1`), reuse this notebook and add `init_scale` as another grouping variable (parsed from run_id or the condition dict).
